# 🏠 Linear Regression — California Housing Dataset
### Atlas Skilltech University | Applied ML Series

---

## Dataset Ke Baare Mein

**California Housing Dataset** — 1990 US Census se liya gaya.

| Feature | Kya Hai? | Type |
|---------|----------|------|
| `MedInc` | Median household income | Numeric |
| `HouseAge` | Median house age in years | Numeric |
| `AveRooms` | Average rooms per household | Numeric |
| `AveBedrms` | Average bedrooms per household | Numeric |
| `Population` | Block population | Numeric |
| `AveOccup` | Average occupancy per household | Numeric |
| `Latitude` | Latitude coordinate | Numeric |
| `Longitude` | Longitude coordinate | Numeric |
| **`MedHouseVal`** | **Target: Median house value ($100K)** | **Numeric** |

**20,640 samples | 8 features | Real-world data**

**Goal:** Predict `MedHouseVal` (house price in $100K) from other features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score, PredictionErrorDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Libraries load ho gayi!')

---
## Step 1: Data Load aur EDA

In [ ]:
# Dataset load karo
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print('=== Dataset Overview ===')
print(f'Shape: {df.shape}')
print(f'Target: MedHouseVal (house price in $100K units)')
print(f'Price Range: ${df.MedHouseVal.min()*100:.0f}K — ${df.MedHouseVal.max()*100:.0f}K')
print()
print(df.head())

In [ ]:
# Basic Stats
print('=== Descriptive Statistics ===')
print(df.describe().round(2))
print(f'\nMissing Values: {df.isnull().sum().sum()}')

In [ ]:
# EDA Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

# Target distribution
axes[0].hist(df.MedHouseVal, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('House Price Distribution')
axes[0].set_xlabel('Median House Value ($100K)')
axes[0].set_ylabel('Count')
axes[0].axvline(df.MedHouseVal.mean(), color='red', linestyle='--',
                label=f'Mean: ${df.MedHouseVal.mean()*100:.0f}K')
axes[0].legend()

# Income vs Price
sample_idx = np.random.choice(len(df), 2000, replace=False)
axes[1].scatter(df.MedInc.iloc[sample_idx], df.MedHouseVal.iloc[sample_idx],
                alpha=0.3, s=10, color='coral')
axes[1].set_xlabel('Median Income ($10K)')
axes[1].set_ylabel('House Value ($100K)')
axes[1].set_title('Income vs House Price')

# House Age distribution
axes[2].hist(df.HouseAge, bins=30, color='green', edgecolor='white', alpha=0.8)
axes[2].set_title('House Age Distribution')
axes[2].set_xlabel('House Age (Years)')

# Latitude vs Price (geography)
sc = axes[3].scatter(df.Longitude.iloc[sample_idx], df.Latitude.iloc[sample_idx],
                     c=df.MedHouseVal.iloc[sample_idx], cmap='hot_r',
                     alpha=0.5, s=5)
plt.colorbar(sc, ax=axes[3], label='Price ($100K)')
axes[3].set_xlabel('Longitude')
axes[3].set_ylabel('Latitude')
axes[3].set_title('Geographic Price Map (California)')

# AveRooms vs Price
df_clipped = df[df.AveRooms < 20]  # Remove extreme outliers
axes[4].scatter(df_clipped.AveRooms, df_clipped.MedHouseVal,
                alpha=0.3, s=5, color='purple')
axes[4].set_xlabel('Average Rooms')
axes[4].set_ylabel('House Value ($100K)')
axes[4].set_title('Rooms vs Price')

# Population distribution
axes[5].hist(np.log1p(df.Population), bins=40, color='orange', edgecolor='white', alpha=0.8)
axes[5].set_title('Population Distribution (Log Scale)')
axes[5].set_xlabel('log(Population)')

plt.suptitle('California Housing — Exploratory Data Analysis', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, linewidths=0.5,
            annot_kws={'size': 10})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()
print(f'\nMedHouseVal se sabse zyada correlated: {corr.MedHouseVal.drop("MedHouseVal").abs().idxmax()}')
print(f'Correlation value: {corr.MedHouseVal.drop("MedHouseVal").abs().max():.3f}')

---
## Step 2: Preprocessing

In [ ]:
# Features aur Target separate karo
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on train
X_test_scaled = scaler.transform(X_test)         # only transform on test

print('=== Preprocessing Complete ===')
print(f'Training samples: {X_train.shape[0]:,}')
print(f'Testing samples:  {X_test.shape[0]:,}')
print(f'Features: {X_train.shape[1]}')
print(f'\nAfter scaling — Mean ≈ 0, Std ≈ 1 (verify):')
print(f'Train mean: {X_train_scaled.mean():.6f}')
print(f'Train std:  {X_train_scaled.std():.6f}')

---
## Step 3: Baseline — Simple Linear Regression (1 Feature)

In [ ]:
# Pehle simple regression — sirf MedInc se predict karo
X_simple_train = X_train[['MedInc']]
X_simple_test = X_test[['MedInc']]

simple_lr = LinearRegression()
simple_lr.fit(X_simple_train, y_train)
y_pred_simple = simple_lr.predict(X_simple_test)

print('=== Simple Linear Regression (1 Feature: MedInc) ===')
print(f'Intercept: {simple_lr.intercept_:.4f}')
print(f'Coefficient: {simple_lr.coef_[0]:.4f}')
print(f'Train R²: {r2_score(y_train, simple_lr.predict(X_simple_train)):.4f}')
print(f'Test R²:  {r2_score(y_test, y_pred_simple):.4f}')
print(f'Test RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_simple))*100:.0f}K')
print(f'\nInterpretation: Har $10K income badhne se house price ~${simple_lr.coef_[0]*100:.0f}K badhta hai')

# Plot
plt.figure(figsize=(10, 6))
sample = np.random.choice(len(X_test), 1000, replace=False)
X_test_sample = X_test['MedInc'].values[sample]
y_test_sample = y_test.values[sample]
plt.scatter(X_test_sample, y_test_sample, alpha=0.3, s=10, color='steelblue', label='Actual')
x_line = np.linspace(X_test['MedInc'].min(), X_test['MedInc'].max(), 100).reshape(-1, 1)
plt.plot(x_line, simple_lr.predict(x_line), 'r-', linewidth=2.5, label='Prediction Line')
plt.xlabel('Median Income ($10K)')
plt.ylabel('Median House Value ($100K)')
plt.title(f'Simple Linear Regression: MedInc → Price | R²={r2_score(y_test, y_pred_simple):.3f}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## Step 4: Multiple Linear Regression (All Features)

In [ ]:
# Multiple Linear Regression
multi_lr = LinearRegression()
multi_lr.fit(X_train_scaled, y_train)
y_pred_multi = multi_lr.predict(X_test_scaled)

print('=== Multiple Linear Regression (8 Features) ===')
print(f'Train R²: {r2_score(y_train, multi_lr.predict(X_train_scaled)):.4f}')
print(f'Test R²:  {r2_score(y_test, y_pred_multi):.4f}')
print(f'Test MAE:  ${mean_absolute_error(y_test, y_pred_multi)*100:.0f}K')
print(f'Test RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_multi))*100:.0f}K')

print('\nCoefficients (Standardized):')
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': multi_lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

# Coefficient bar chart
plt.figure(figsize=(10, 5))
colors = ['green' if c > 0 else 'red' for c in coef_df.Coefficient]
plt.barh(coef_df.Feature, coef_df.Coefficient, color=colors, alpha=0.8, edgecolor='black')
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Standardized Coefficient')
plt.title('Feature Importance — Multiple Linear Regression Coefficients')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5: Residual Analysis

In [ ]:
residuals = y_test.values - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Predicted (Method 1 — Manual)
axes[0, 0].scatter(y_pred_multi, residuals, alpha=0.3, s=10, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Value ($100K)')
axes[0, 0].set_ylabel('Residual')
axes[0, 0].set_title('Residuals vs Predicted (Method 1: Manual)')
axes[0, 0].grid(alpha=0.3)

# Plot 2: Residual Histogram
axes[0, 1].hist(residuals, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Residual')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Residual Distribution')
axes[0, 1].grid(alpha=0.3)

# Plot 3: Actual vs Predicted (Method 2 — PredictionErrorDisplay)
PredictionErrorDisplay.from_predictions(
    y_test, y_pred_multi,
    kind='actual_vs_predicted',
    ax=axes[1, 0],
    scatter_kwargs={'alpha': 0.3, 's': 5}
)
axes[1, 0].set_title('Actual vs Predicted (Method 2: PredictionErrorDisplay)')

# Plot 4: seaborn residplot
sample_idx = np.random.choice(len(y_pred_multi), 1000, replace=False)
sns.residplot(
    x=y_pred_multi[sample_idx], y=y_test.values[sample_idx],
    lowess=True,
    scatter_kws={'alpha': 0.3, 's': 10},
    line_kws={'color': 'red', 'linewidth': 2},
    ax=axes[1, 1]
)
axes[1, 1].set_xlabel('Predicted ($100K)')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residual Plot with LOWESS (seaborn)')
axes[1, 1].grid(alpha=0.3)

plt.suptitle('Residual Analysis — California Housing', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('💡 Note: Price 5 par capped hai (data censoring) → funnel at high prices.')

---
## Step 6: Regularization Comparison — Ridge vs Lasso

In [ ]:
# GridSearchCV for Best Alpha
print('=== GridSearchCV for Regularization ===')

alphas_grid = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_gs = GridSearchCV(
    Ridge(), {'alpha': alphas_grid},
    cv=5, scoring='r2'
)
ridge_gs.fit(X_train_scaled, y_train)

lasso_gs = GridSearchCV(
    Lasso(), {'alpha': [0.0001, 0.001, 0.01, 0.1, 1]},
    cv=5, scoring='r2'
)
lasso_gs.fit(X_train_scaled, y_train)

print(f'Ridge Best Alpha: {ridge_gs.best_params_["alpha"]} | CV R²: {ridge_gs.best_score_:.4f}')
print(f'Lasso Best Alpha: {lasso_gs.best_params_["alpha"]} | CV R²: {lasso_gs.best_score_:.4f}')

# Final comparison
models_final = {
    'Linear Regression': multi_lr,
    f'Ridge (α={ridge_gs.best_params_["alpha"]})': ridge_gs.best_estimator_,
    f'Lasso (α={lasso_gs.best_params_["alpha"]})': lasso_gs.best_estimator_,
}

print('\n=== Final Test Performance ===')
for name, mdl in models_final.items():
    y_p = mdl.predict(X_test_scaled)
    print(f'{name:25s}: R²={r2_score(y_test, y_p):.4f}, '
          f'RMSE=${np.sqrt(mean_squared_error(y_test, y_p))*100:.0f}K')

In [ ]:
# Ridge vs Lasso Coefficient Paths
alphas_path = np.logspace(-3, 3, 100)
ridge_coefs, lasso_coefs = [], []

for a in alphas_path:
    r = Ridge(alpha=a).fit(X_train_scaled, y_train)
    ridge_coefs.append(r.coef_)
    l = Lasso(alpha=a, max_iter=5000).fit(X_train_scaled, y_train)
    lasso_coefs.append(l.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
feature_names = X.columns
colors_feat = plt.cm.Set1(np.linspace(0, 1, len(feature_names)))

for i, (name, color) in enumerate(zip(feature_names, colors_feat)):
    axes[0].semilogx(alphas_path, ridge_coefs[:, i], color=color, label=name, linewidth=1.5)
    axes[1].semilogx(alphas_path, lasso_coefs[:, i], color=color, label=name, linewidth=1.5)

for ax, title in [(axes[0], 'Ridge (L2) — Coefficients shrink to 0 gradually'),
                   (axes[1], 'Lasso (L1) — Coefficients go EXACTLY to 0')]:
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('Alpha (Regularization Strength)')
    ax.set_ylabel('Coefficient Value')
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Coefficient Regularization Paths — Ridge vs Lasso', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 7: Cross Validation aur Model Deployment

In [ ]:
# 5-Fold CV on best model
best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=ridge_gs.best_params_['alpha']))
])

cv_r2 = cross_val_score(best_pipeline, X, y, cv=5, scoring='r2')
cv_rmse = cross_val_score(best_pipeline, X, y, cv=5,
                           scoring='neg_root_mean_squared_error')

print('=== 5-Fold Cross Validation — Best Model ===')
print(f'R² per fold:   {cv_r2.round(4)}')
print(f'Mean R²:       {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'RMSE per fold: {(-cv_rmse * 100).round(0)}')
print(f'Mean RMSE:     ${(-cv_rmse).mean()*100:.0f}K ± ${(-cv_rmse).std()*100:.0f}K')

# CV fold visualization
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_r2, color='steelblue', edgecolor='black', alpha=0.8)
plt.axhline(cv_r2.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean R² = {cv_r2.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('5-Fold Cross Validation — Ridge Regression')
plt.legend()
plt.ylim(0.5, 0.75)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Model Deploy karo — Save with joblib
import joblib

# Final pipeline train karo on full data
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=ridge_gs.best_params_['alpha']))
])
final_pipeline.fit(X, y)  # Train on ALL data

joblib.dump(final_pipeline, '/tmp/california_housing_model.pkl')
print('✅ Model saved!')

# Predict on new data
new_house = pd.DataFrame({
    'MedInc': [5.0],
    'HouseAge': [20.0],
    'AveRooms': [5.0],
    'AveBedrms': [1.0],
    'Population': [1000.0],
    'AveOccup': [3.0],
    'Latitude': [34.0],
    'Longitude': [-118.0]
})

loaded_model = joblib.load('/tmp/california_housing_model.pkl')
prediction = loaded_model.predict(new_house)[0]
print(f'\nNaya Ghar Predict:  ${prediction * 100:.0f}K')
print(f'(MedInc=5, Age=20, LA area, 5 rooms)')

---
## Capstone Project

### Yeh tumhara final challenge hai! 🏆

Sab kuch khud try karo — answers notebook ke end mein hain.

### Part A: Feature Engineering Challenge

**Task:** Create 2 new features aur dekho R² improve hota hai ya nahi.

**Hint features:**
- `RoomsPerPerson = AveRooms / AveOccup`
- `BedroomRatio = AveBedrms / AveRooms`
- Income brackets as categorical

In [ ]:
# Part A: Feature Engineering — aap khud karein!
df_eng = df.copy()

# TODO: Naye features banao
# df_eng['RoomsPerPerson'] = ?
# df_eng['BedroomRatio'] = ?

# TODO: Model train karo new features ke saath
# X_new = df_eng.drop('MedHouseVal', axis=1)
# ... train, evaluate, compare with baseline

print('💡 Hint: RoomsPerPerson = AveRooms / AveOccup')
print('Feature engineering often boosts R² by 2-5% on this dataset!')

### Part B: Regional Analysis

**Task:** California ko geography ke basis par regions mein divide karo aur ek model per region banao.

**Hint:**
- Latitude > 37.5 → Northern California
- Latitude <= 37.5 → Southern California

In [ ]:
# Part B: Regional Analysis
df_north = df[df.Latitude > 37.5]
df_south = df[df.Latitude <= 37.5]

print(f'Northern California: {len(df_north):,} samples')
print(f'Southern California: {len(df_south):,} samples')

# TODO: Har region ke liye alag model banao
# north_model = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
# south_model = ...
# Compare R² for both regions vs single global model

print('\n💡 Hypothesis: Regional models better perform karenge? Test karo!')

### Part C: ElasticNet Hyperparameter Search

**Task:** GridSearchCV se ElasticNet ke liye best `alpha` aur `l1_ratio` dhundho.

In [ ]:
# Part C: ElasticNet GridSearchCV
param_grid_en = {
    'alpha': [0.001, 0.01, 0.1, 1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

# TODO: GridSearchCV implement karo
# en_gs = GridSearchCV(ElasticNet(), param_grid_en, cv=5, scoring='r2')
# en_gs.fit(X_train_scaled, y_train)
# print(f'Best params: {en_gs.best_params_}')
# print(f'Best CV R²: {en_gs.best_score_:.4f}')

print('💡 Hint: l1_ratio=1.0 = Lasso, l1_ratio=0.0 = Ridge')

### Part D: Error Analysis — Worst Predictions

**Task:** Worst 20 predictions dhundho aur analyze karo kahan model fail kar raha hai.

In [ ]:
# Part D: Error Analysis
y_pred_analysis = final_pipeline.predict(X_test)
errors = np.abs(y_test.values - y_pred_analysis)

error_df = X_test.copy()
error_df['Actual'] = y_test.values
error_df['Predicted'] = y_pred_analysis
error_df['AbsError'] = errors

print('=== Top 10 Worst Predictions ===')
print(error_df.nlargest(10, 'AbsError')[['MedInc', 'HouseAge', 'Latitude',
                                          'Actual', 'Predicted', 'AbsError']].to_string())

# TODO: Pattern dhundho
# - Kya worst errors sirf high-price houses pe hain?
# - Kya specific latitude/longitude pe errors zyada hain?
# - Kya income group pe depends karta hai?

print('\n💡 Analyze: Kya error ek jagah cluster ho raha hai?')

---
## Final Summary

| Metric | Simple LR | Multiple LR | Ridge | Lasso |
|--------|-----------|-------------|-------|-------|
| **Features Used** | 1 (MedInc) | 8 | 8 | 8 |
| **Test R²** | ~0.47 | ~0.60 | ~0.61 | ~0.60 |
| **Test RMSE** | ~$80K | ~$70K | ~$70K | ~$70K |

**Key Learnings:**
1. **More features** usually improve R² (simple → multiple LR jump)
2. **Regularization** zyada improvement nahi deta jab multicollinearity kam ho
3. **Geography matters** — spatial features (Latitude/Longitude) important hain
4. **Price capping** (at $500K in 1990 data) creates residual patterns at high end
5. **Feature engineering** (RoomsPerPerson etc.) kaafi boost de sakta hai

**Next Steps:**
- Random Forest Regressor try karo (non-linear relationships capture karega)
- Polynomial features add karo
- Spatial clustering features banana